In [1]:
import psycopg2
%store -r DB_CONFIG

# Cell 1: สร้างตารางตาม Schema ที่ออกแบบ 
def create_tables():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        
        # คำสั่งสร้างตาราง (SQL)
        query = """
        CREATE TABLE IF NOT EXISTS stock_data (
            id SERIAL PRIMARY KEY,
            symbol VARCHAR(10),
            price DECIMAL(10, 2),
            change_percent DECIMAL(10, 2),
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
        """
        cur.execute(query)
        conn.commit()
        cur.close()
        conn.close()
        print("✅ 2. สร้าง Table และ Schema สำเร็จ (หัวข้อ 3.1)")
    except Exception as e:
        print(f"❌ Error: {e}")

create_tables()
%store create_tables

✅ 2. สร้าง Table และ Schema สำเร็จ (หัวข้อ 3.1)
Proper storage of interactively declared classes (or instances
of those classes) is not possible! Only instances
of classes in real modules on file system can be %store'd.



In [2]:
def insert_stock_data(symbol, price, change_percent):
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        
        insert_query = """
        INSERT INTO stock_data (symbol, price, change_percent)
        VALUES (%s, %s, %s)
        """
        cur.execute(insert_query, (symbol, price, change_percent))
        
        conn.commit()
        cur.close()
        conn.close()
    except Exception as e:
        print(f"❌ Error inserting data: {e}")

# เก็บฟังก์ชันไว้ใช้ข้ามไฟล์
%store insert_stock_data

Proper storage of interactively declared classes (or instances
of those classes) is not possible! Only instances
of classes in real modules on file system can be %store'd.



In [3]:
# 2.4 - 2.5 เชื่อมต่อกับ PostgreSQL
import psycopg2

try:
    # ดึงค่าคอนฟิกที่เคย %store ไว้จากไฟล์ที่ 1
    %store -r DB_CONFIG
    
    # ทดลองเชื่อมต่อ
    conn = psycopg2.connect(**DB_CONFIG)
    print("✅ 2.5 การเชื่อมต่อกับฐานข้อมูลสำเร็จ (Connected to PostgreSQL)")
    conn.close()
except Exception as e:
    print(f"❌ 2.5 พบข้อผิดพลาดในการเชื่อมต่อ: {e}")

✅ 2.5 การเชื่อมต่อกับฐานข้อมูลสำเร็จ (Connected to PostgreSQL)


In [4]:
# 2.6 ตรวจสอบหรือสร้าง Database (ขั้นตอนนี้ปกติทำผ่าน pgAdmin ไปแล้ว)
print("✅ 2.6 ตรวจสอบฐานข้อมูล 'financial_db' เรียบร้อยแล้ว")

✅ 2.6 ตรวจสอบฐานข้อมูล 'financial_db' เรียบร้อยแล้ว


In [5]:
# 2.7 สร้างตาราง stock_prices
def create_stock_table():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        
        # สร้างตารางที่เก็บราคาหุ้นรายวัน
        query = """
        CREATE TABLE IF NOT EXISTS stock_prices (
            id SERIAL PRIMARY KEY,
            ticker VARCHAR(10),
            date DATE,
            open_price DECIMAL(10, 2),
            high_price DECIMAL(10, 2),
            low_price DECIMAL(10, 2),
            close_price DECIMAL(10, 2),
            volume BIGINT,
            UNIQUE(ticker, date)
        );
        """
        cur.execute(query)
        conn.commit()
        cur.close()
        conn.close()
        print("✅ 2.7 สร้างตาราง stock_prices สำเร็จ!")
    except Exception as e:
        print(f"❌ 2.7 Error: {e}")

create_stock_table()

✅ 2.7 สร้างตาราง stock_prices สำเร็จ!


In [15]:
import pandas as pd
import psycopg2

target_files = ['PTT_real.csv', 'KBANK_real.csv', 'ADVANC_real.csv', 'AOT_real.csv', 'BDMS_real.csv']

try:
    %store -r DB_CONFIG
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    
    for file in target_files:
        df = pd.read_csv(file)
        ticker_name = file.split('_')[0]
        
        # ล้างชื่อคอลัมน์ให้เป็นตัวพิมพ์เล็กทั้งหมดเพื่อหาคอลง่ายๆ
        df.columns = [c.strip().lower() for c in df.columns]
        
        for _, row in df.iterrows():
            try:
                # 1. หาค่า Volume (หาจากชื่อ 'volume' ถ้าไม่เจอให้ลองหาตำแหน่งที่ 6 หรือ 5)
                vol_raw = 0
                if 'volume' in df.columns:
                    vol_raw = row['volume']
                else:
                    # ถ้าหาชื่อไม่เจอ ให้ดูว่ามีคอลัมน์สุดท้ายไหม และต้องไม่ใช่ค่าที่มี %
                    for val in reversed(row.values):
                        if isinstance(val, (int, float)) or (isinstance(val, str) and '%' not in val):
                            vol_raw = val
                            break
                
                # ล้างพวกคอมม่าออก
                vol_clean = str(vol_raw).replace(',', '').replace('nan', '0')
                volume = int(float(vol_clean)) if vol_clean.strip() != '' else 0
                
                # 2. เตรียมข้อมูล (ดึงตามลำดับพื้นฐาน: Date, Open, High, Low, Close)
                # เราใช้ .iloc เพื่อความชัวร์ในตำแหน่งราคาหลัก
                query = """
                    INSERT INTO stock_prices (ticker, date, open_price, high_price, low_price, close_price, volume)
                    VALUES (%s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (ticker, date) DO NOTHING;
                """
                cur.execute(query, (
                    ticker_name, 
                    row.iloc[0], # Date
                    row.iloc[1], # Open
                    row.iloc[2], # High
                    row.iloc[3], # Low
                    row.iloc[4], # Close
                    volume
                ))
            except Exception:
                continue # แถวไหนพังให้ข้ามไปเลย ไม่ต้องหยุดโปรแกรม
        
        print(f"📊 นำเข้า {ticker_name} เรียบร้อย!")

    conn.commit() 
    cur.close()
    conn.close()
    print("\n✅ [2.8] เรียบร้อย! ข้อมูลถูก Commit ลง Database แล้วครับเพื่อน")

except Exception as e:
    print(f"❌ พังหนัก: {e}")

📊 นำเข้า PTT เรียบร้อย!
📊 นำเข้า KBANK เรียบร้อย!
📊 นำเข้า ADVANC เรียบร้อย!
📊 นำเข้า AOT เรียบร้อย!
📊 นำเข้า BDMS เรียบร้อย!

✅ [2.8] เรียบร้อย! ข้อมูลถูก Commit ลง Database แล้วครับเพื่อน


In [1]:
import pandas as pd
import psycopg2

target_files = ['PTT_real.csv', 'KBANK_real.csv', 'ADVANC_real.csv', 'AOT_real.csv', 'BDMS_real.csv']

try:
    %store -r DB_CONFIG
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    
    # --- ขั้นตอนที่ 1: ล้างข้อมูลเก่าในตารางออกให้หมดก่อน (เพื่อความชัวร์) ---
    cur.execute("DELETE FROM stock_prices;")
    conn.commit()
    print("🧹 ล้างข้อมูลเดิมในตารางเรียบร้อย...")

    # --- ขั้นตอนที่ 2: เริ่มนำเข้าใหม่ ---
    for file in target_files:
        try:
            df = pd.read_csv(file)
            ticker_name = file.split('_')[0]
            
            count = 0
            for _, row in df.iterrows():
                # ดึงข้อมูลตามลำดับ: 0=Date, 1=Open, 2=High, 3=Low, 4=Close, 6=Volume
                vol_raw = str(row.iloc[6]).replace(',', '').replace('%', '') if len(row) > 6 else '0'
                volume = int(float(vol_raw)) if vol_raw.strip() != '' else 0
                
                query = """
                    INSERT INTO stock_prices (ticker, date, open_price, high_price, low_price, close_price, volume)
                    VALUES (%s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (ticker, date) DO NOTHING;
                """
                cur.execute(query, (ticker_name, row.iloc[0], row.iloc[1], row.iloc[2], row.iloc[3], row.iloc[4], volume))
                count += 1
            
            # Commit ทุกครั้งที่จบ 1 หุ้น
            conn.commit()
            print(f"✅ นำเข้า {ticker_name} สำเร็จ {count} แถว และบันทึกแล้ว")
            
        except Exception as e:
            print(f"❌ ไฟล์ {file} มีปัญหา: {e}")
            conn.rollback() # ถ้าพังให้ถอยกลับมาตั้งหลัก

    cur.close()
    conn.close()
    print("\n🚀 [DONE] ข้อมูลทั้งหมดถูกยืนยันลง Database แล้ว!")

except Exception as e:
    print(f"❌ เชื่อมต่อ DB ไม่ได้: {e}")

🧹 ล้างข้อมูลเดิมในตารางเรียบร้อย...
✅ นำเข้า PTT สำเร็จ 1279 แถว และบันทึกแล้ว
✅ นำเข้า KBANK สำเร็จ 1279 แถว และบันทึกแล้ว
✅ นำเข้า ADVANC สำเร็จ 1279 แถว และบันทึกแล้ว
✅ นำเข้า AOT สำเร็จ 1279 แถว และบันทึกแล้ว
✅ นำเข้า BDMS สำเร็จ 1279 แถว และบันทึกแล้ว

🚀 [DONE] ข้อมูลทั้งหมดถูกยืนยันลง Database แล้ว!
